# Amazon Review Alignment: A100 Formal Pipeline

使用 `Qwen/Qwen3.5-2B`、BF16 与 4-bit QLoRA 执行正式的五模型实验。
该配置控制了 PPO/GRPO prompt 数与评估规模，但 Colab Compute Unit
消耗是动态的，不能保证固定在 100 CU 内。


## 1. 挂载 Drive 并加载项目

本 Notebook 将仓库放在 Google Drive，训练输出和 checkpoint 会在断开
Colab 后保留。目标 GPU：`A100`。

开始前必须先将本地最新代码和本 Notebook 提交并推送到 GitHub `main`
分支，否则下方 `git clone` 会获取旧版本。


In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/j156734119/Amazon-reviews-2023-electronics-SFT-DPO.git"
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--short"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if status:
        print("Preserving Colab-local tracked changes before pull:")
        print(status)
        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "stash",
                "push",
                "-m",
                "colab-auto-stash-before-pull",
            ],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

## 2. 安装依赖

执行后使用 Colab 菜单 **运行时 -> 重新启动会话**。重启后从下一单元格
继续，不需要再次执行安装。


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
    check=True,
)
print("Installation complete. Restart the Colab runtime now.")


## 3. 重启后恢复目录、加载 Secrets

在 Colab 左侧钥匙图标中添加 `OPENAI_API_KEY`。`HF_TOKEN` 对公开模型
是可选的，但能提高 Hugging Face 下载限额。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
os.chdir(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
CONFIG = "configs/rlhf_a100.yaml"

try:
    import amazon_review_alignment
    import bitsandbytes
    import peft
    import transformers
    import trl
except (ImportError, ModuleNotFoundError):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".[train,eval,dev]",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    import amazon_review_alignment
    import bitsandbytes
    import peft
    import transformers
    import trl

for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[secret_name] = value

def cli(*arguments: str, check: bool = True) -> subprocess.CompletedProcess:
    command = [
        sys.executable,
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]
    print("\n$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    returncode = process.wait()
    result = subprocess.CompletedProcess(
        command,
        returncode,
        stdout="".join(output_lines),
        stderr=None,
    )
    if check and returncode:
        raise RuntimeError(
            f"Command failed with exit code {returncode}: "
            + " ".join(command)
        )
    return result

print("Config:", CONFIG)
print("Package:", Path(amazon_review_alignment.__file__).resolve())
print(
    "Training stack:",
    transformers.__version__,
    trl.__version__,
    peft.__version__,
    bitsandbytes.__version__,
)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))


## A100 环境检查


In [ ]:
import importlib.metadata

import torch
import transformers
import trl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a Colab GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {total_gib:.2f} GiB")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", importlib.metadata.version("peft"))

if "A100".lower() not in gpu_name.lower():
    raise RuntimeError("Expected A100, but Colab assigned: " + gpu_name)
if total_gib < 38:
    raise RuntimeError("Insufficient GPU memory for this profile.")
if True and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This profile requires BF16 support.")


## 4. 选择 A100 Mini Smoke 或正式训练

首次运行保持 `RUN_MODE="mini"`。它使用 Qwen3.5-2B 和真实 A100
装配，但只训练一步，并使用 60 条评论验证完整链路。全部通过后改成
`RUN_MODE="formal"`，重新从数据准备开始执行正式实验。

Mini 与正式输出目录完全隔离，不会相互复用 checkpoint。


In [ ]:
import yaml

from amazon_review_alignment.config import load_config

RUN_MODE = "formal"  # mini | formal

if RUN_MODE == "mini":
    merged = load_config(REPO_DIR / "configs" / "rlhf_a100.yaml")
    merged.pop("_config_path", None)

    old_root = "outputs/a100-qwen3.5-2b"
    new_root = "outputs/a100-mini-qwen3.5-2b"

    def replace_output_paths(value):
        if isinstance(value, dict):
            return {
                key: replace_output_paths(item)
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [replace_output_paths(item) for item in value]
        if isinstance(value, str):
            return value.replace(old_root, new_root)
        return value

    merged = replace_output_paths(merged)
    merged["project"]["output_dir"] = new_root
    merged["data"].update(
        {
            "sample_size": 60,
            "max_scanned_reviews": 20000,
            "rating_targets": {
                "1": 12,
                "2": 12,
                "3": 12,
                "4": 12,
                "5": 12,
            },
            "splits": {
                "train": 42,
                "validation": 6,
                "test": 12,
            },
        }
    )
    merged["teacher"].update(
        {
            "pilot_size": 5,
            "max_estimated_cost_usd": 1.0,
        }
    )
    merged["training"]["sft"]["max_steps"] = 1
    merged["training"]["dpo"]["max_steps"] = 1
    merged["rlhf"].update(
        {
            "human_calibration_samples": 0,
            "ai_reward_train_pairs": 4,
            "ai_reward_validation_pairs": 2,
            "ppo_prompt_count": 4,
        }
    )
    merged["rlhf"]["reward"]["max_steps"] = 1
    merged["rlhf"]["ppo"]["total_episodes"] = 4
    merged["rlhf"]["ppo"]["gradient_accumulation_steps"] = 1
    merged["rlhf"]["ppo"]["save_steps"] = 1
    merged["rlhf"]["grpo"]["prompt_count"] = 4
    merged["rlhf"]["grpo"]["max_steps"] = 1
    merged["evaluation"]["max_test_samples"] = 4

    mini_path = Path("/content/rlhf_a100_mini.yaml")
    mini_path.write_text(
        yaml.safe_dump(merged, sort_keys=False),
        encoding="utf-8",
    )
    CONFIG = str(mini_path)
    effective = merged
elif RUN_MODE == "formal":
    CONFIG = "configs/rlhf_a100.yaml"
    effective = load_config(REPO_DIR / CONFIG)
else:
    raise ValueError("RUN_MODE must be 'mini' or 'formal'.")

print("Run mode:", RUN_MODE)
print("Effective config:", CONFIG)
print(
    "PPO auxiliary models in 4-bit:",
    effective["rlhf"]["ppo"]["auxiliary_model_load_in_4bit"],
)


## 4. 测试、准备数据并生成 Base baseline


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=REPO_DIR,
    check=True,
)
cli("prepare-data", "--config", CONFIG)


In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "--force-inference",
)

import pandas as pd
from amazon_review_alignment.config import load_config

output_root = Path(
    load_config(CONFIG)["project"]["output_dir"]
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))


## 5. 教师数据

Pilot 会立即调用 OpenAI API。Batch 提交后可能需要等待；提交成功后可以
关闭 GPU Runtime，稍后重新连接并重复“检查 Batch”单元格。


In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
cli("teacher-pilot", "--config", CONFIG)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive, userdata

drive.mount("/content/drive")

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
CONFIG = "configs/rlhf_a100.yaml"

# 进入项目目录，保证相对配置路径有效
os.chdir(REPO_DIR)

# 加载 Colab Secrets
for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None

    if value:
        os.environ[secret_name] = value

# 重新安装当前项目源码
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

def cli(*arguments, check=True):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]

    print("\n$", " ".join(command), flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    return_code = process.wait()

    if check and return_code:
        print("\n===== LAST 200 LINES =====")
        print("".join(lines[-200:]))
        raise RuntimeError(
            f"Command failed with exit code {return_code}"
        )

    return subprocess.CompletedProcess(
        command,
        return_code,
        stdout="".join(lines),
    )

print("Repository:", REPO_DIR)
print("Config:", CONFIG)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))

In [ ]:
# 第一次执行会提交 Batch；后续重复执行会查询并下载结果。
cli("teacher-batch", "--config", CONFIG)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
CONFIG = str(REPO_DIR / "configs" / "rlhf_a100.yaml")

if not REPO_DIR.exists():
    raise RuntimeError(f"项目目录不存在：{REPO_DIR}")

# 加载 Secrets
for name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None

    if value:
        os.environ[name] = value

# 重新安装本地项目
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

# 确保当前 Python 进程能立即找到 src package
src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.chdir(REPO_DIR)

from amazon_review_alignment.config import load_config

def cli(*arguments, check=True):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(arg) for arg in arguments),
    ]

    print("\n$", " ".join(command), flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    return_code = process.wait()

    if check and return_code:
        print("".join(lines[-200:]))
        raise RuntimeError(
            f"Command failed with exit code {return_code}"
        )

    return subprocess.CompletedProcess(
        command,
        return_code,
        stdout="".join(lines),
    )

print("Repository:", REPO_DIR)
print("Config:", CONFIG)
print("Package loaded successfully")
print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
from pathlib import Path

from amazon_review_alignment.config import load_config

effective_config = load_config(CONFIG)
output_root = (
    REPO_DIR / effective_config["project"]["output_dir"]
).resolve()

train_preferences = (
    output_root / "teacher" / "preferences_train.jsonl"
)
validation_preferences = (
    output_root / "teacher" / "preferences_validation.jsonl"
)

print("Output root:", output_root)
print("Train file exists:", train_preferences.exists())
print(
    "Validation file exists:",
    validation_preferences.exists(),
)

# 文件不存在时，查询已有 Batch 并尝试下载结果
if not train_preferences.exists() or not validation_preferences.exists():
    cli("teacher-batch", "--config", CONFIG)

if not train_preferences.exists() or not validation_preferences.exists():
    raise RuntimeError(
        "OpenAI Batch 尚未完成。可以断开 A100，稍后重新运行本单元格。"
    )

train_rows = sum(
    1 for line in train_preferences.open(encoding="utf-8")
    if line.strip()
)
validation_rows = sum(
    1 for line in validation_preferences.open(encoding="utf-8")
    if line.strip()
)

print("Teacher train rows:", train_rows)
print("Teacher validation rows:", validation_rows)

if train_rows == 0 or validation_rows == 0:
    raise RuntimeError("Teacher 数据文件存在，但没有有效记录。")

print("Teacher data ready. 可以开始 SFT。")

## 6. SFT、合并权重与 DPO


In [ ]:
cli("train-sft", "--config", CONFIG)


In [ ]:
cli("merge-sft", "--config", CONFIG)


In [ ]:
cli("train-dpo", "--config", CONFIG)


In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "sft",
    "dpo",
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))


In [ ]:
from pathlib import Path
import pandas as pd

output_root = (
    REPO_DIR
    / "outputs"
    / "a100-qwen3.5-2b"
)

for name in ("sft", "sft-merged", "dpo"):
    path = output_root / "models" / name
    print(name, path.exists(), path)

for variant in ("base", "sft", "dpo"):
    path = (
        output_root
        / "evaluation"
        / "predictions"
        / f"{variant}.jsonl"
    )

    rows = 0
    if path.exists():
        rows = sum(
            1 for line in path.open(encoding="utf-8")
            if line.strip()
        )

    print(variant, "predictions:", rows)

metrics_path = output_root / "evaluation" / "metrics.csv"
display(pd.read_csv(metrics_path))

In [ ]:
from pathlib import Path
import json
import pandas as pd

output_root = (
    REPO_DIR
    / "outputs"
    / "a100-qwen3.5-2b"
)
prediction_dir = output_root / "evaluation" / "predictions"

def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]

sft_rows = read_jsonl(prediction_dir / "sft.jsonl")
dpo_rows = read_jsonl(prediction_dir / "dpo.jsonl")

metrics = pd.read_csv(
    output_root / "evaluation" / "metrics.csv"
)
display(metrics)

print("SFT rows:", len(sft_rows))
print("DPO rows:", len(dpo_rows))

# 展示前 10 个 DPO 输出
for index, (sft, dpo) in enumerate(
    zip(sft_rows, dpo_rows),
    start=1,
):
    if index > 10:
        break

    print("\n" + "=" * 100)
    print("INDEX:", index)
    print("REVIEW:", dpo["text"][:800])
    print("\nSFT:")
    print(sft["raw_output"])
    print("\nDPO:")
    print(dpo["raw_output"])

In [ ]:
from pathlib import Path
import json

dpo_dir = output_root / "models" / "dpo"
sft_merged_dir = output_root / "models" / "sft-merged"

print("DPO directory:", dpo_dir)
print("DPO exists:", dpo_dir.exists())
print("Merged SFT exists:", sft_merged_dir.exists())

for path in sorted(dpo_dir.iterdir()):
    print(path.name, path.stat().st_size)

adapter_config = dpo_dir / "adapter_config.json"

if adapter_config.exists():
    print("\nDPO adapter config:")
    print(
        json.dumps(
            json.loads(
                adapter_config.read_text(encoding="utf-8")
            ),
            indent=2,
        )
    )

In [ ]:
from pathlib import Path
import json
import pandas as pd

state_path = (
    output_root
    / "models"
    / "dpo"
    / "checkpoint-65"
    / "trainer_state.json"
)

state = json.loads(state_path.read_text(encoding="utf-8"))
history = pd.DataFrame(state["log_history"])

display(history)
history.to_csv(
    output_root / "evaluation" / "dpo_training_history.csv",
    index=False,
)

In [ ]:
from pathlib import Path
import yaml

from amazon_review_alignment.config import load_config

# 加载并合并 full.yaml 与 rlhf_a100.yaml
config = load_config(str(CONFIG))
config.pop("_config_path", None)

config["training"]["dpo"].update(
    {
        "learning_rate": 1e-5,
        "epochs": 1,
        "output_dir": (
            "outputs/a100-qwen3.5-2b/models/dpo-v2"
        ),
    }
)

DPO_V2_CONFIG = Path("/content/rlhf_a100_dpo_v2.yaml")
DPO_V2_CONFIG.write_text(
    yaml.safe_dump(
        config,
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)

print("DPO v2 config:", DPO_V2_CONFIG)
print("Learning rate:", config["training"]["dpo"]["learning_rate"])
print("Epochs:", config["training"]["dpo"]["epochs"])
print("Output:", config["training"]["dpo"]["output_dir"])

In [ ]:
cli(
    "train-dpo",
    "--config",
    str(DPO_V2_CONFIG),
)

In [ ]:
from pathlib import Path
import shutil

prediction_dir = (
    REPO_DIR
    / "outputs"
    / "a100-qwen3.5-2b"
    / "evaluation"
    / "predictions"
)

old_prediction = prediction_dir / "dpo.jsonl"
backup_prediction = prediction_dir / "dpo-v1.jsonl"

if old_prediction.exists() and not backup_prediction.exists():
    shutil.copy2(old_prediction, backup_prediction)
    print("Backed up:", backup_prediction)

cli(
    "inference",
    "--config",
    str(DPO_V2_CONFIG),
    "--variant",
    "dpo",
    "--force",
)

## 7. 构建纯 RLAIF 数据

A100 正式流程不要求人工填写 200 条 A/B。Reward Model、PPO 和 GRPO
直接使用 OpenAI 教师生成并通过规则校验的 chosen/rejected 偏好。

这属于 RLAIF，而不是纯 RLHF。独立的 200 条人工盲评仅用于最终评估，
不进入训练数据。


In [ ]:
cli("build-rlhf-data", "--config", CONFIG)

import json

manifest_path = output_root / "rlhf" / "data_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))
assert manifest["alignment_method"] == "rlaif"
assert manifest["human_total_rows"] == 0


## 8. Reward Model、PPO 与 GRPO

每个阶段是独立单元格。阶段失败时先处理报错，不要跳过并继续。


In [ ]:
cli("train-reward", "--config", CONFIG)


In [ ]:
cli("train-ppo", "--config", CONFIG)


In [ ]:
cli("train-grpo", "--config", CONFIG)


## 9. 五模型统一评估和报告


In [4]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount("/content/drive", force_remount=True)

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)

print("MyDrive exists:", Path("/content/drive/MyDrive").exists())
print("Workspace exists:", REPO_DIR.parent.exists())
print("Repository exists:", REPO_DIR.exists())

if REPO_DIR.parent.exists():
    print(
        "Workspace contents:",
        [path.name for path in REPO_DIR.parent.iterdir()],
    )

if not REPO_DIR.exists():
    raise RuntimeError(
        "Drive已挂载，但repo目录确实不存在。"
        "请检查Drive回收站或是否移动了amazon-review-alignment-workspace。"
    )

os.chdir(REPO_DIR)

source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

print("Recovered repository:", Path.cwd())

Mounted at /content/drive
MyDrive exists: True
Workspace exists: True
Repository exists: True
Workspace contents: ['repo']
Recovered repository: /content/drive/MyDrive/amazon-review-alignment-workspace/repo


In [8]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
OUTPUT_ROOT = REPO_DIR / "outputs" / "a100-qwen3.5-2b"
CONFIG_V2 = (
    OUTPUT_ROOT
    / "run_configs"
    / "rlhf_a100_dpo_v2.yaml"
)

os.chdir(REPO_DIR)

# 将当前项目安装到此Colab运行时
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

# 修复4-bit推理依赖
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "bitsandbytes>=0.46.1",
    ],
    check=True,
)

grpo_adapter = (
    OUTPUT_ROOT
    / "models"
    / "grpo"
    / "adapter_model.safetensors"
)

print("Config exists:", CONFIG_V2.exists())
print("GRPO adapter exists:", grpo_adapter.exists())

if not CONFIG_V2.exists():
    raise RuntimeError(f"配置不存在：{CONFIG_V2}")

if not grpo_adapter.exists():
    raise RuntimeError(f"GRPO adapter不存在：{grpo_adapter}")

# 显式向子进程传入src目录
env = os.environ.copy()
src_path = str(REPO_DIR / "src")
old_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    src_path
    if not old_pythonpath
    else src_path + os.pathsep + old_pythonpath
)
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"

# 验证子进程可以导入项目
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import amazon_review_alignment; "
            "print('Package:', amazon_review_alignment.__file__)"
        ),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

command = [
    sys.executable,
    "-u",
    "-m",
    "amazon_review_alignment.cli",
    "inference",
    "--config",
    str(CONFIG_V2),
    "--variant",
    "grpo",
]

print("\n$", " ".join(command), flush=True)

process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

lines = []
assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)
    lines.append(line)

return_code = process.wait()
print("\nReturn code:", return_code)

if return_code != 0:
    print("\n===== GRPO ERROR: LAST 200 LINES =====")
    print("".join(lines[-200:]))
    raise RuntimeError("GRPO推理失败")

grpo_prediction = (
    OUTPUT_ROOT
    / "evaluation"
    / "predictions"
    / "grpo.jsonl"
)
rows = sum(1 for _ in grpo_prediction.open(encoding="utf-8"))

print("GRPO推理完成")
print("Prediction:", grpo_prediction)
print("Rows:", rows)

Config exists: True
GRPO adapter exists: True

$ /usr/bin/python3 -u -m amazon_review_alignment.cli inference --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/rlhf_a100_dpo_v2.yaml --variant grpo
2026-06-15 13:29:34,771 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 1/320 [00:10<54:24, 10.23s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 320/320 [00:36<00:00,  8.73it/s]
/usr/local/l

In [9]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
OUTPUT_ROOT = REPO_DIR / "outputs" / "a100-qwen3.5-2b"
CONFIG_V2 = (
    OUTPUT_ROOT
    / "run_configs"
    / "rlhf_a100_dpo_v2.yaml"
)
PREDICTION_DIR = OUTPUT_ROOT / "evaluation" / "predictions"

os.chdir(REPO_DIR)

env = os.environ.copy()
src_path = str(REPO_DIR / "src")
env["PYTHONPATH"] = (
    src_path + os.pathsep + env["PYTHONPATH"]
    if env.get("PYTHONPATH")
    else src_path
)
env["PYTHONUNBUFFERED"] = "1"

def cli(*arguments):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(arg) for arg in arguments),
    ]

    print("\n$", " ".join(command), flush=True)

    process = subprocess.run(
        command,
        cwd=REPO_DIR,
        env=env,
        text=True,
    )

    if process.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {process.returncode}"
        )

# 验证五种预测完整，不会重新推理
expected_variants = ["base", "sft", "dpo", "ppo", "grpo"]

for variant in expected_variants:
    path = PREDICTION_DIR / f"{variant}.jsonl"

    if not path.exists():
        raise RuntimeError(f"缺少预测文件：{path}")

    rows = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"{variant:5}: {rows}/500")

    if rows != 500:
        raise RuntimeError(
            f"{variant}预测不完整：{rows}/500"
        )

# 使用已有预测计算指标，不添加--force-inference
cli(
    "evaluate",
    "--config",
    CONFIG_V2,
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
)

cli(
    "build-report",
    "--config",
    CONFIG_V2,
)

metrics_path = OUTPUT_ROOT / "evaluation" / "metrics.csv"
summary_path = (
    OUTPUT_ROOT / "evaluation" / "evaluation_summary.json"
)
report_path = OUTPUT_ROOT / "report.md"

print("\n===== FINAL METRICS =====")
display(pd.read_csv(metrics_path))

print("\nMetrics:", metrics_path)
print("Summary:", summary_path)
print("Report:", report_path)

base : 500/500
sft  : 500/500
dpo  : 500/500
ppo  : 500/500
grpo : 500/500

$ /usr/bin/python3 -u -m amazon_review_alignment.cli evaluate --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/rlhf_a100_dpo_v2.yaml --variants base sft dpo ppo grpo

$ /usr/bin/python3 -u -m amazon_review_alignment.cli build-report --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/rlhf_a100_dpo_v2.yaml

===== FINAL METRICS =====


,variant,examples,schema_valid_rate,evidence_grounded_rate,word_limit_ok_rate,instruction_following_rate
0,base,500,0.954,0.842,0.954,0.842
1,sft,500,1.000,0.960,1.000,0.960
2,dpo,500,0.846,0.800,0.846,0.800
3,ppo,500,1.000,0.954,1.000,0.954
4,grpo,500,1.000,0.950,1.000,0.950



Metrics: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/metrics.csv
Summary: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/evaluation_summary.json
Report: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/report.md


In [3]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
OUTPUT_ROOT = REPO_DIR / "outputs" / "a100-qwen3.5-2b"
PREDICTION_DIR = OUTPUT_ROOT / "evaluation" / "predictions"
CONFIG_V2 = (
    OUTPUT_ROOT / "run_configs" / "rlhf_a100_dpo_v2.yaml"
)

if not REPO_DIR.exists():
    raise RuntimeError(f"仓库不存在：{REPO_DIR}")

if not CONFIG_V2.exists():
    raise RuntimeError(f"配置不存在：{CONFIG_V2}")

# 检查五个预测是否完整
for variant in ["base", "sft", "dpo", "ppo", "grpo"]:
    path = PREDICTION_DIR / f"{variant}.jsonl"
    rows = (
        sum(1 for _ in path.open(encoding="utf-8"))
        if path.exists()
        else 0
    )
    print(f"{variant:5}: exists={path.exists()}, rows={rows}")

    if rows != 500:
        raise RuntimeError(f"{variant} 预测不完整：{rows}/500")

# 让子进程能够找到src包
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "src")
env["PYTHONUNBUFFERED"] = "1"

def cli(*arguments):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(arg) for arg in arguments),
    ]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(
        command,
        cwd=REPO_DIR,
        env=env,
        check=True,
    )

# 只读取已有预测并计算指标，不重新推理
cli(
    "evaluate",
    "--config",
    CONFIG_V2,
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
)

cli("build-report", "--config", CONFIG_V2)

metrics_path = OUTPUT_ROOT / "evaluation" / "metrics.csv"
report_path = OUTPUT_ROOT / "report.md"

display(pd.read_csv(metrics_path))
print("Metrics:", metrics_path)
print("Report:", report_path)

Mounted at /content/drive
base : exists=True, rows=500
sft  : exists=True, rows=500
dpo  : exists=True, rows=500
ppo  : exists=True, rows=500
grpo : exists=True, rows=500

$ /usr/bin/python3 -u -m amazon_review_alignment.cli evaluate --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/rlhf_a100_dpo_v2.yaml --variants base sft dpo ppo grpo

$ /usr/bin/python3 -u -m amazon_review_alignment.cli build-report --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/rlhf_a100_dpo_v2.yaml


,variant,examples,schema_valid_rate,evidence_grounded_rate,word_limit_ok_rate,instruction_following_rate
0,base,500,0.954,0.842,0.954,0.842
1,sft,500,1.000,0.960,1.000,0.960
2,dpo,500,0.846,0.800,0.846,0.800
3,ppo,500,1.000,0.954,1.000,0.954
4,grpo,500,1.000,0.950,1.000,0.950


Metrics: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/metrics.csv
Report: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/report.md


In [4]:
from pathlib import Path
import json
import pandas as pd

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
    "/outputs/a100-qwen3.5-2b"
)

metrics = pd.read_csv(
    OUTPUT_ROOT / "evaluation" / "metrics.csv"
).set_index("variant")

# 相对Base提升，单位为百分点
base = metrics.loc["base"]
delta_vs_base = (
    metrics.drop(columns=["examples"])
    .subtract(base.drop(labels=["examples"]))
    .multiply(100)
    .round(1)
)

print("===== 相对 Base 的提升（百分点）=====")
display(delta_vs_base)

# 相对SFT提升，适合比较DPO/PPO/GRPO
sft = metrics.loc["sft"]
delta_vs_sft = (
    metrics.loc[["dpo", "ppo", "grpo"]]
    .drop(columns=["examples"])
    .subtract(sft.drop(labels=["examples"]))
    .multiply(100)
    .round(1)
)

print("===== 相对 SFT 的提升（百分点）=====")
display(delta_vs_sft)

# 显示Reward Model、PPO、GRPO训练指标
for filename in [
    "reward_metrics.json",
    "ppo_metrics.json",
    "grpo_metrics.json",
]:
    path = OUTPUT_ROOT / "rlhf" / filename
    print(f"\n===== {filename} =====")
    if path.exists():
        display(json.loads(path.read_text(encoding="utf-8")))
    else:
        print("文件不存在")

===== 相对 Base 的提升（百分点）=====


,schema_valid_rate,evidence_grounded_rate,word_limit_ok_rate,instruction_following_rate
variant,,,,
base,0.0,0.0,0.0,0.0
sft,4.6,11.8,4.6,11.8
dpo,-10.8,-4.2,-10.8,-4.2
ppo,4.6,11.2,4.6,11.2
grpo,4.6,10.8,4.6,10.8


===== 相对 SFT 的提升（百分点）=====


,schema_valid_rate,evidence_grounded_rate,word_limit_ok_rate,instruction_following_rate
variant,,,,
dpo,-15.4,-16.0,-15.4,-16.0
ppo,0.0,-0.6,0.0,-0.6
grpo,0.0,-1.0,0.0,-1.0



===== reward_metrics.json =====


{'ai_validation': {'examples': 130,
  'preference_accuracy': 0.9,
  'mean_reward_margin': 2.3178386981670673},
 'human_held_out': {'examples': 0,
  'preference_accuracy': 0.0,
  'mean_reward_margin': 0.0}}


===== ppo_metrics.json =====


{'episodes': 128,
 'unique_prompts': 128,
 'runtime_seconds': 1412.0017797649998,
 'peak_cuda_memory_allocated_gb': 13.48516321182251,
 'peak_cuda_memory_reserved_gb': 13.693359375,
 'final_logged_metrics': {'objective/kl': -0.16860847175121307,
  'objective/rlhf_reward': 4.414680480957031,
  'objective/scores': 4.40625,
  'loss/policy_avg': 0.006341753527522087,
  'loss/value_avg': 5.725080966949463},
 'reference_policy': 'merged SFT policy with PPO adapter disabled',
 'auxiliary_model_load_in_4bit': False,
 'auxiliary_model_dtype': 'torch.bfloat16',
 'model_device_reports': {'policy': {'parameter_devices': ['cuda:0'],
   'parameter_dtypes': ['torch.bfloat16', 'torch.float32', 'torch.uint8'],
   'buffer_devices': ['cuda:0'],
   'embedding_device': 'cuda:0',
   'embedding_dtype': 'torch.float32',
   'score_device': None,
   'score_dtype': None},
  'reward': {'parameter_devices': ['cuda:0'],
   'parameter_dtypes': ['torch.bfloat16'],
   'buffer_devices': ['cuda:0'],
   'embedding_device


===== grpo_metrics.json =====


{'unique_prompts': 128,
 'num_generations': 4,
 'expected_completions_per_epoch': 512,
 'runtime_seconds': 2284.579908508,
 'peak_cuda_memory_allocated_gb': 7.7075066566467285,
 'peak_cuda_memory_reserved_gb': 8.421875,
 'final_logged_metrics': {'completions/mean_length': 59.75,
  'completions/clipped_ratio': 0.0,
  'rewards/sft-merged/mean': -0.444732666015625,
  'rewards/sft-merged/std': 0.32063624262809753,
  'rewards/schema_reward/mean': 1.0,
  'rewards/schema_reward/std': 0.0,
  'rewards/evidence_reward/mean': 1.0,
  'rewards/evidence_reward/std': 0.0,
  'rewards/length_reward/mean': 1.0,
  'rewards/length_reward/std': 0.0,
  'reward': 2.305267333984375,
  'reward_std': 0.32063621282577515,
  'frac_reward_zero_std': 0.0,
  'kl': 0.000557376530196052,
  'entropy': 0.33097050338983536,
  'clip_ratio/region_mean': 0.0},
 'reference_policy': 'merged SFT policy with GRPO adapter disabled',
 'reward_weights': {'reward_model': 1.0,
  'schema': 1.0,
  'evidence': 1.5,
  'length': 0.25},
 

In [5]:
from pathlib import Path
import os
import subprocess
import sys
import yaml
import pandas as pd

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
OUTPUT_ROOT = REPO_DIR / "outputs" / "a100-qwen3.5-2b"
BASE_CONFIG = (
    OUTPUT_ROOT / "run_configs" / "rlhf_a100_dpo_v2.yaml"
)

os.chdir(REPO_DIR)

if not os.getenv("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Colab Secrets中没有OPENAI_API_KEY")

# 读取现有DPO v2配置，创建Judge专用配置
config = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))

config["evaluation"]["run_llm_judge"] = True
config["evaluation"]["judge_samples_per_pair"] = 50
config["evaluation"]["judge_pairs"] = [
    ["base", "sft"],
    ["sft", "dpo"],
    ["sft", "ppo"],
    ["sft", "grpo"],
    ["dpo", "ppo"],
    ["dpo", "grpo"],
    ["ppo", "grpo"],
]

JUDGE_CONFIG = OUTPUT_ROOT / "run_configs" / "llm_judge.yaml"
JUDGE_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "src")
env["PYTHONUNBUFFERED"] = "1"

def cli(*args):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(arg) for arg in args),
    ]
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=REPO_DIR, env=env, check=True)

# 不加force-inference，复用已有500条预测
cli(
    "evaluate",
    "--config",
    JUDGE_CONFIG,
    "--llm-judge",
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
)

cli("build-report", "--config", JUDGE_CONFIG)

summary_path = (
    OUTPUT_ROOT
    / "evaluation"
    / "judge_pairwise_summary.csv"
)

print("\n===== LLM JUDGE RESULTS =====")
display(pd.read_csv(summary_path))

print(
    "Report:",
    OUTPUT_ROOT / "evaluation" / "report.md",
)


$ /usr/bin/python3 -u -m amazon_review_alignment.cli evaluate --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/llm_judge.yaml --llm-judge --variants base sft dpo ppo grpo

$ /usr/bin/python3 -u -m amazon_review_alignment.cli build-report --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/llm_judge.yaml

===== LLM JUDGE RESULTS =====


,comparison,examples,left_model,right_model,right_model_win_rate_ties_half,ci_95_low,ci_95_high,ties
0,base_vs_sft,50,base,sft,0.57,0.44,0.70,3
1,sft_vs_dpo,50,sft,dpo,0.75,0.64,0.85,7
2,sft_vs_ppo,50,sft,ppo,0.41,0.31,0.52,19
3,sft_vs_grpo,50,sft,grpo,0.48,0.37,0.59,20
4,dpo_vs_ppo,50,dpo,ppo,0.36,0.25,0.48,10
5,dpo_vs_grpo,50,dpo,grpo,0.40,0.28,0.52,12
6,ppo_vs_grpo,50,ppo,grpo,0.46,0.41,0.50,44


Report: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/report.md


In [6]:
cli(
    "human-eval",
    "--config",
    JUDGE_CONFIG,
    "--samples",
    "200",
    "--left-variant",
    "ppo",
    "--right-variant",
    "grpo",
)


$ /usr/bin/python3 -u -m amazon_review_alignment.cli human-eval --config /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/run_configs/llm_judge.yaml --samples 200 --left-variant ppo --right-variant grpo


## 11. 扩展 PPO/GRPO v2 实验

原实验的 128 个 prompt 是资源受限 feasibility baseline，并非 5,000
条评论全部进入在线 RL。5,000 条原始评论被切分为 3,500 train、500
validation 和 1,000 test；PPO/GRPO 只能使用 train，且必须避开 Reward
Model 训练样本和 test。

v2 从原始 train 中抽取 1,024 个与 RM 严格不重叠的共享 prompt，是原
baseline 的 8 倍。PPO/GRPO 均从相同 SFT policy、Reward Model 和 prompt
IDs 开始，分别保存到 `ppo-v2`、`grpo-v2`，不覆盖旧 adapter。

预计 A100 40GB 总耗时约 10–14 小时。每个训练阶段是独立单元，完成后
checkpoint 和 adapter 都会保存在 Drive。


In [ ]:
from pathlib import Path
import json
import shutil

from amazon_review_alignment.config import load_config

ONLINE_V2_CONFIG = "configs/rlhf_a100_online_v2.yaml"
online_v2 = load_config(REPO_DIR / ONLINE_V2_CONFIG)
online_root = Path(online_v2["project"]["output_dir"]).resolve()
archive_dir = online_root / "archive" / "online-v1"
archive_dir.mkdir(parents=True, exist_ok=True)

files_to_archive = [
    "rlhf/data_manifest.json",
    "rlhf/ppo_prompts.jsonl",
    "rlhf/grpo_prompts.jsonl",
    "rlhf/ppo_metrics.json",
    "rlhf/grpo_metrics.json",
    "rlhf/ppo_log_history.json",
    "rlhf/grpo_log_history.json",
    "evaluation/metrics.csv",
    "evaluation/evaluation_summary.json",
    "evaluation/report.md",
    "evaluation/judge_decisions.jsonl",
    "evaluation/judge_pairwise_summary.csv",
    "evaluation/predictions/ppo.jsonl",
    "evaluation/predictions/grpo.jsonl",
]
for relative in files_to_archive:
    source = online_root / relative
    target = archive_dir / relative
    if source.exists() and not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)

cli("build-rlhf-data", "--config", ONLINE_V2_CONFIG)

manifest_path = online_root / "rlhf" / "data_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["ppo_prompts"] == 1024
assert manifest["grpo_prompts"] == 1024
assert manifest["ppo_grpo_shared_prompt_ids"] is True
assert (
    manifest["online_prompt_source"]
    == "raw_train_excluded_from_reward_model"
)
print(json.dumps(manifest, indent=2))
print("Archived v1 artifacts:", archive_dir.resolve())


### 11.1 PPO v2

训练 1,024 episodes，温度提高到 0.9 增加探索，KL 系数从 0.05
降至 0.02，避免策略被过强地固定在 SFT 附近。


In [ ]:
cli("train-ppo", "--config", ONLINE_V2_CONFIG)


### 11.2 GRPO v2

使用同一批 1,024 prompts，每条生成 4 个候选，共约 4,096 个
completions。提高 RM 权重并降低已饱和规则奖励权重，`beta=0.01`
允许比 v1 更充分的策略更新。


In [ ]:
cli("train-grpo", "--config", ONLINE_V2_CONFIG)


### 11.3 v2 推理与硬指标


In [ ]:
cli(
    "inference",
    "--config",
    ONLINE_V2_CONFIG,
    "--variant",
    "ppo",
    "--force",
)
cli(
    "inference",
    "--config",
    ONLINE_V2_CONFIG,
    "--variant",
    "grpo",
    "--force",
)

prediction_dir = online_root / "evaluation" / "predictions"
shutil.copy2(
    prediction_dir / "ppo.jsonl",
    prediction_dir / "ppo-v2.jsonl",
)
shutil.copy2(
    prediction_dir / "grpo.jsonl",
    prediction_dir / "grpo-v2.jsonl",
)

cli(
    "evaluate",
    "--config",
    ONLINE_V2_CONFIG,
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
)
display(pd.read_csv(online_root / "evaluation" / "metrics.csv"))


### 11.4 五模型全两两 AI 盲评

对 Base、SFT、DPO、PPO v2、GRPO v2 的 10 种组合各抽取 100 条，
共 1,000 次匿名 A/B/tie 判断。若中断，响应哈希会确保只继续缺失或
已改变的模型回答，不会把 v1 的 PPO/GRPO 判断误用于 v2。


In [ ]:
cli(
    "evaluate",
    "--config",
    ONLINE_V2_CONFIG,
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
    "--llm-judge",
    "--judge-samples-per-pair",
    "100",
)
cli("build-report", "--config", ONLINE_V2_CONFIG)

judge_path = (
    online_root
    / "evaluation"
    / "judge_pairwise_summary.csv"
)
print("===== ONLINE V2 AI BLIND JUDGE =====")
display(pd.read_csv(judge_path))
print(
    "Report:",
    (online_root / "evaluation" / "report.md").resolve(),
)
